In [1]:
import yaml

In [2]:
from anndata import read_h5ad

In [3]:
from os.path import join

In [4]:
import h5py

In [5]:
import pandas as pd

In [6]:
sample_group_pairs = [
    # AKI vs. HRT
    ('EnrollmentCategory', ('Healthy Reference', 'AKI')),
    # AKI vs. H-CKD. (H-CKD not in enrollment category values anymore. Should I use "Hypertension History" Yes/No column?)
    ('EnrollmentCategory', ('AKI', 'CKD')),
    # D-CKD vs. HRT. (D-CKD not in enrollment category values anymore. Should I use "Diabetes History" Yes/No column?)
    ('EnrollmentCategory', ('CKD', 'Healthy Reference')),
    # Diabetes CKD vs. Hypertension CKD. (DKD nor H-CKD not in enrollment category values anymore. Should I use Yes/No columns?)
    #('EnrollmentCategory', ('DKD', 'H-CKD')),
    # D-CKD vs. HRT
    ('AdjudicatedCategory', ('Diabetic Kidney Disease', 'Healthy Reference')),
    # Acute tubular injury vs. HRT
    ('AdjudicatedCategory', ('Acute Tubular Injury', 'Healthy Reference')),
    # Acute interstitial nephritis vs. HRT
    ('AdjudicatedCategory', ('Acute Interstitial Nephritis', 'Healthy Reference')),
    # Diabetes CKD vs. Hypertension CKD
    ('AdjudicatedCategory', ('Diabetic Kidney Disease', 'Hypertensive Kidney Disease')),
    # ATN vs. AIN
    ('AdjudicatedCategory', ('Acute Interstitial Nephritis', 'Acute Tubular Injury')),

    # TODO: use Diabetes History and Hypertension History columns here.
]
cell_type_cols = [
    "subclass_l1",
    "subclass_l2",
]

In [7]:
adata_path = join("data", "raw", "kpmp-sc-aug-2026", "KPMP_PREMIERE_SC_version2_ForExplorer_RemovedBatchEffect_Final2025.h5ad")
clinical_path = join("data", "raw", "kpmp-sc-aug-2026", "20260618_OpenAccessClinicalData.csv")

sc_id_mapping_path = join("data", "raw", "kpmp-sc-aug-2026", "sc_left_join_clinical_NBON_08.12.2026.xlsx")

In [8]:
adata = read_h5ad(adata_path)

In [9]:
adata

AnnData object with n_obs × n_vars = 348984 × 33298
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'KPMPID', 'barcode', 'SpecimenID', 'LibraryID', 'InternalID', 'Tissue.Type', 'Protocol', 'Age', 'Gender', 'Race', 'assay', 'tissue', 'organism', 'disease', 'SampleID', 'umap_1', 'umap_2', 'cell_type', 'development_stage', 'development_stage_ontology_term_id', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'cell_type_ontology_term_id', 'DataSourceID', 'state.l2', 'state.l1', 'class', 'SubclassLevel2_FullName', 'subclass.level1', 'subclass.level2', 'cluster', 'EnrollmentCategory', 'AdjudicationCategory', 'self_reported_race', 'diabetes_history', 'hypertension_history', 'condition'
    var: 'vf_vst_counts_mean', 'vf_vst_counts_variance', 'vf_vst_counts_variance.expected', 'vf_vst_counts_variance.standardized', 'vf_vst_counts_variable', 'vf_vst_counts_rank', 'var.features', 'var.features.rank'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'

In [10]:
# Fix categorical columns
f = h5py.File(adata_path)

def clean_category_chars(s):
    # The cell types have odd characters
    return s.replace("ï", "i").replace("⁺", "_pos").replace("ʰⁱ", "_hi")
    
def fix_categorical_column(colname):
    if adata.obs[colname].dtype.kind == "i":
        # This is an integer column
        try:
            categories = f[f"/obs/__categories/{colname}"][()].astype(str)
        except:
            # UnicodeDecodeError
            categories = [ b.decode("utf-8") for b in f[f"/obs/__categories/{colname}"][()] ]

        categories = [ clean_category_chars(c) for c in categories ]
        return adata.obs[colname].apply(lambda i: categories[i])

adata.obs["subclass.level1"] = fix_categorical_column("subclass.level1")
adata.obs["subclass.level2"] = fix_categorical_column("subclass.level2")

adata.obs = adata.obs.rename(columns={"subclass.level1": "subclass_l1", "subclass.level2": "subclass_l2"})

In [11]:
# Read in xlsx file with the SpecimenID to Participant ID mapping
# Append a new column to adata.obs with the Participant ID value corresponding to each adata.obs specimen value.

id_map_df = pd.read_excel(sc_id_mapping_path, sheet_name="sample to participant mapping")
id_map_df = id_map_df.set_index("SpecimenID")

In [12]:
id_map_df.head()

,KPMP Participant ID,Unnamed: 2
SpecimenID,,
S-2008-000628-HRT,163-2,NaN
S-2008-000605-HRT,163-3,NaN
S-2008-000651-HRT,163-4,NaN
S-2106-003885-HRT,163-5,NaN
S-2109-023336-HRT,163-7,NaN


In [13]:
adata.obs["KPMP Participant ID"] = adata.obs["SpecimenID"].apply(lambda specimen_id: id_map_df.at[specimen_id, "KPMP Participant ID"])

In [14]:
kpmp_id_to_exclude = "30-11051"

In [15]:
adata.shape

(348984, 33298)

In [16]:
should_include_kpmp_id = adata.obs["KPMPID"] != kpmp_id_to_exclude
adata = adata[should_include_kpmp_id, :].copy()
adata.shape

(346388, 33298)

In [17]:
clean_adata_path = join("data", "raw", "kpmp-sc-aug-2026", "KPMP_PREMIERE_SC_version2_ForExplorer_RemovedBatchEffect_Final2025.clean.h5ad")
# Un-comment to write the cleaned anndata object
# adata.write_h5ad(clean_adata_path)

In [18]:
# Investigate clinical data to identify the adata.obs column which matches the clinical "Participant ID" values
clinical_data = pd.read_csv(clinical_path)

In [19]:
# Determine unique values for the clinical sample group columns
clinical_data.columns

Index(['Participant ID', 'Tissue Source', 'Protocol', 'Sample Type',
       'Enrollment Category', 'Primary Adjudicated Category', 'Sex',
       'Age (Years) (Binned)', 'Race', 'KDIGO Stage',
       'Baseline eGFR (ml/min/1.73m2)',
       'Baseline eGFR (ml/min/1.73m2) (Binned)', 'Proteinuria (mg) (Binned)',
       'A1c (%) (Binned)', 'Albuminuria (mg) (Binned)', 'Diabetes History',
       'Diabetes Duration (Years)', 'Hypertension History',
       'Hypertension Duration (Years)', 'On RAAS Blockade'],
      dtype='object')

In [21]:
clinical_data["Enrollment Category"].unique().tolist()

['Healthy Reference', 'CKD', 'DM-R', 'AKI', 'FSGS']

In [22]:
clinical_data["Sample Type"].unique().tolist()

['Intra-operative Biopsy',
 'Transplant Pre-perfusion Biopsy',
 'Percutaneous Needle Biopsy',
 'Intra-operative Needle Biopsy',
 'Deceased Donor Nephrectomy',
 'Partial Tumor Nephrectomy',
 'Total Tumor Nephrectomy',
 'Tumor Nephrectomy',
 'Biopsy',
 'Transplant Surveillance Biopsy']

In [23]:
clinical_data["Primary Adjudicated Category"].unique().tolist()

[nan,
 'Diabetic Kidney Disease',
 'Other',
 'Cannot be determined',
 'Hypertensive Kidney Disease',
 'Acute Interstitial Nephritis',
 'Acute Tubular Injury']

In [ ]:
patient_ids = clinical_data["Participant ID"].unique().tolist()
len(patient_ids)

In [ ]:
obs_df = adata.obs

In [ ]:
grouped_df = obs_df.groupby(["SampleID", "KPMPID", "InternalID", "LibraryID", "SpecimenID"]).count().reset_index()

In [ ]:
left_joined = grouped_df[["SampleID", "KPMPID", "InternalID", "LibraryID", "SpecimenID"]].merge(clinical_data, left_on="KPMPID", right_on="Participant ID", how="left")

In [ ]:
left_joined[["Participant ID", "SampleID", "KPMPID", "InternalID", "LibraryID", "SpecimenID"]].to_csv("sc_left_join_clinical.csv")

In [ ]:
adata.obs.columns

In [ ]:
# Potential ID cols
adata_kpmp_ids = adata.obs["KPMPID"].unique().tolist()
adata_library_ids = adata.obs["LibraryID"].unique().tolist()
adata_specimen_ids = adata.obs["SpecimenID"].unique().tolist()
adata_internal_ids = adata.obs["InternalID"].unique().tolist()
adata_sample_ids = adata.obs["SampleID"].unique().tolist()

In [ ]:
len(adata_kpmp_ids), len(adata_library_ids), len(adata_specimen_ids), len(adata_internal_ids), len(adata_sample_ids)

In [ ]:
sum([ (adata_val in patient_ids) for adata_val in adata_kpmp_ids ])

In [ ]:
sum([ (adata_val in patient_ids) for adata_val in adata_library_ids ])

In [ ]:
sum([ (adata_val in patient_ids) for adata_val in adata_specimen_ids ])

In [ ]:
sum([ (adata_val in patient_ids) for adata_val in adata_internal_ids ])

In [ ]:
sum([ (adata_val in patient_ids) for adata_val in adata_sample_ids ])

In [ ]:
for id_val in [ adata_val for adata_val in adata_kpmp_ids if (adata_val not in patient_ids) ]:
    print(id_val)

In [ ]:
adata.obs

In [ ]:
adata.obs["subclass_l1"].unique().tolist()

In [ ]:
adata.obs["SubclassLevel2_FullName"].unique().tolist()

In [ ]:
cell_types = {}
for colname in cell_type_cols:
    cell_types[colname] = sorted(adata.obs[colname].unique().tolist(), key=lambda v: v.lower())

In [ ]:
sample_group_pairs_dict = [
    { "colname": t[0], "lhs": t[1][0], "rhs": t[1][1] }
    for t in sample_group_pairs
]

In [ ]:
import pandas as pd
# Join adata.obs with clinical data from CSV
clinical_data = pd.read_csv(clinical_path)

adata.obs = adata.obs.merge(clinical_data, left_on="KPMPID", right_on="Participant ID", how="left")

# We could have done a left join, but then we would have to filter out samples that do not have clinical data later.
# We also do not want strings to be converted to NaN, as these cause Zarr writing errors like "TypeError: expected unicode string, found nan".

# This effectively does an inner join. We cannot use how="inner", since this would only affect adata.obs, and not other anndata fields.
has_clinical_data = ~adata.obs["Participant ID"].isna()
adata = adata[has_clinical_data, :].copy()

In [25]:
patient_ids = adata.obs["KPMP Participant ID"].unique().tolist()
specimen_ids = adata.obs["SpecimenID"].unique().tolist()

In [ ]:
adata.shape

In [26]:
yaml_output = yaml.dump({
    "patient_ids": patient_ids,
    "specimen_ids": specimen_ids,
}, default_flow_style=False)
print(yaml_output)

patient_ids:
- PRE027
- PRE038
- PRE98sc
- PRE018-1
- PRE19
- PRE055-1
- PRE062-1
- REF67
- 34-10050
- 29-10006
- 28-10051
- 27-10039
- 31-10000
- 31-10001
- 30-10034
- 30-10018
- 30-10123
- 33-10005
- 33-10006
- 32-10074
- 32-10003
- 29-10008
- 29-10011
- 31-10013
- 31-10035
- 29-10013
- 29-10010
- 29-10016
- 31-10042
- 27-10088
- 29-10386
- 30-10125
- 30-11080
- 30-11095
- 32-10419
- 32-10458
- 28-12613
- 163-3
- 163-2
- 163-4
- 31-10322
- 27-10146
- 163-5
- 164-6
- 29-10420
- 29-10423
- 31-10340
- 165-9
- 164-10
- 165-14
- 28-12545
- 28-12557
- 34-10478
- 163-7
- 164-13
- 164-15
- 28-12546
- 27-10155
- 27-10185
- 27-10170
- 27-10154
- 27-10165
- 164-19
- 164-20
- 34-10573
- 29-10425
- 164-22
- 934-10001
- Sample1153-EO1
- Sample1153-EO2
- Sample1153-EO3
- Sample1157-EO1
- Sample1157-EO2
- Sample1157-EO3
- Sample1158-EO1
- Sample1158-EO2
- Sample1158-EO3
- Sample1162-EO1
- Sample1162-EO2
- 21-015
- 21-016
- 21-019
- 21-020
- 34-10187
- 34-10184
- 28-12263
- 27-10071
- 32-10034
- 29-1

In [ ]:
import numpy as np
should_subset = True
if should_subset:
    print("SUBSETTING")
    # subset using random sample so that multiple sample groups are represented to enable comparison
    np.random.seed(1)
    obs_subset = np.random.choice(adata.obs.index.tolist(), size=20_000, replace=False).tolist()
    var_slice = slice(0, 5_000)
    adata = adata[obs_subset, var_slice].copy()

In [ ]:
# CLEANUP FROM SCRIPT
import pandas as pd
# Join adata.obs with clinical data from CSV
clinical_data = pd.read_csv(clinical_path)

adata.obs = adata.obs.merge(clinical_data, left_on="patient", right_on="Participant ID", how="left")

# We could have done a left join, but then we would have to filter out samples that do not have clinical data later.
# We also do not want strings to be converted to NaN, as these cause Zarr writing errors like "TypeError: expected unicode string, found nan".

# This effectively does an inner join. We cannot use how="inner", since this would only affect adata.obs, and not other anndata fields.
has_clinical_data = ~adata.obs["Participant ID"].isna()
adata = adata[has_clinical_data, :].copy()

#print(adata.obs.head())

adata.obs["Primary Adjudicated Category"] = adata.obs["Primary Adjudicated Category"].fillna("NA")

# Cleanup of sample-level data
def clean_adjudicated_category(row):
    if row["Primary Adjudicated Category"] != "NA":
        return row["Primary Adjudicated Category"]
    else:
        # The row was empty, so perhaps this sample has not yet been adjudicated.
        # However, we also need to check that this was not a "Healthy Reference" sample,
        # as these never go through the adjudication process.
        if row["Enrollment Category"] in ["Healthy Reference"]:
            return "Healthy Reference"
        return ""
adata.obs["AdjudicatedCategory"] = adata.obs.apply(clean_adjudicated_category, axis='columns')
adata.obs["EnrollmentCategory"] = adata.obs["Enrollment Category"]

# TODO: process other clinical columns? Sex, age group, etc.

adata.obs = adata.obs.rename(columns={"subclass.l1": "subclass_l1", "subclass.l2": "subclass_l2", "subclass.l3": "subclass_l3"})

for colname in adata.obs.columns:
    if pd.api.types.is_string_dtype(adata.obs[colname]) or str(adata.obs[colname].dtype) == "object":
        print(f"Filling NAs in string column {colname} with 'NA'")
        adata.obs[colname] = adata.obs[colname].fillna("NA")
    else:
        print(f"Not filling NAs in non-string column {colname} of type {adata.obs[colname].dtype}")

# Column names cannot contain slashes
adata.obs = adata.obs.rename(columns=dict(zip(adata.obs.columns, [c.replace("/", " per ") for c in adata.obs.columns])))


In [ ]:
adata.layers['counts']

In [ ]:
import decoupler as dc

In [ ]:
# References:
# - https://pertpy.readthedocs.io/en/stable/tutorials/notebooks/differential_gene_expression.html#pseudobulks
# - https://decoupler.readthedocs.io/en/latest/api/generated/decoupler.pp.pseudobulk.html

In [ ]:
pdata = dc.pp.pseudobulk(adata, sample_col="specimen", groups_col="subclass_l1", layer="counts", mode="sum", empty=True, verbose=False)

In [ ]:
del adata

In [ ]:
pdata

In [ ]:
import pertpy as pt

In [ ]:
pdata.obs["subclass_l1"].unique()

In [ ]:
pdata.obs["specimen"].unique()

In [ ]:
18*160

In [ ]:
(pdata.obs['psbulk_cells'] != 0.0).sum()

In [ ]:
(pdata.obs['psbulk_counts'] != 0.0).sum()

In [ ]:
(pdata.obs['psbulk_cells'] == 0.0).sum()

In [ ]:
(pdata.obs['psbulk_counts'] == 0.0).sum()

In [ ]:
pdata[0,:]

In [ ]:
pdata.X.shape

In [ ]:
conditions_with_nonzero_expression = (pdata.obs['psbulk_counts'] != 0.0)

In [ ]:
genes_to_keep = pdata.X.sum(axis=0) >= 10

In [ ]:
nonzero_pdata = pdata[conditions_with_nonzero_expression, genes_to_keep].copy()

In [ ]:
nonzero_pdata.shape

In [ ]:
pds2 = pt.tl.PyDESeq2(adata=nonzero_pdata, design=f"~subclass_l1")
pds2.fit(n_cpus=4)

In [ ]:
df = pds2.test_contrasts(pds2.contrast(column="subclass_l1", baseline="POD", group_to_compare=""))